In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.44.2 datasets==2.19.0

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found ex

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/AncientRusProject_SpanCollator_V1"
TOKENIZER_DIR = f"{BASE_DIR}/ancient_rus_tokenizer"
DATA_FILE = f"{BASE_DIR}/ancient_rus_ready_for_bert.txt"
MODEL_DIR = f"{BASE_DIR}/mini_bert_ancient_rus"

In [ ]:
import torch
import random
import math
from datasets import load_dataset
from transformers import (
    BertConfig, BertForMaskedLM, BertTokenizerFast,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback
)

In [ ]:
dataset = load_dataset("text", data_files={"train": DATA_FILE})
tokenizer = BertTokenizerFast.from_pretrained(TOKENIZER_DIR)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
split_dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=False)

In [ ]:
tokenized_datasets = split_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [ ]:
def group_texts(examples):
    block_size = 256
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    return result

In [ ]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [ ]:
config = BertConfig(
    vocab_size=len(tokenizer),
    hidden_size=512,          # 🔥 Увеличили в 2 раза (было 256)
    num_hidden_layers=6,      # 🔥 Добавили слоев (было 4)
    num_attention_heads=8,    # 🔥 Увеличили в 2 раза (было 4)
    intermediate_size=2048,   # 🔥 Увеличили (было 1024)
    max_position_embeddings=512,
    pad_token_id=tokenizer.pad_token_id,
)

In [ ]:
model = BertForMaskedLM(config)
print(f"🧠 Параметры модели: {model.num_parameters():,} (Оптимизировано для быстрого обучения)")

🧠 Параметры модели: 34,833,202 (Оптимизировано для быстрого обучения)


In [ ]:
class SpanDataCollator(DataCollatorForLanguageModeling):
    def __init__(self, tokenizer, mlm_probability=0.15, max_span_length=3):
        super().__init__(tokenizer=tokenizer, mlm=True, mlm_probability=mlm_probability)
        self.max_span_length = max_span_length

    def torch_mask_tokens(self, inputs, special_tokens_mask=None):
        labels = inputs.clone()
        probability_matrix = torch.full(labels.shape, self.mlm_probability)

        if special_tokens_mask is None:
            special_tokens_mask = [
                self.tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
            ]
            special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        else:
            special_tokens_mask = special_tokens_mask.bool()

        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
        masked_indices = torch.bernoulli(probability_matrix).bool()

        span_masked_indices = masked_indices.clone()
        for i in range(labels.shape[0]):
            for j in range(labels.shape[1]):
                if masked_indices[i, j]:
                    span_len = random.randint(1, self.max_span_length)
                    end_idx = min(j + span_len, labels.shape[1])
                    if not special_tokens_mask[i, j:end_idx].any():
                        span_masked_indices[i, j:end_idx] = True

        labels[~span_masked_indices] = -100

        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & span_masked_indices
        inputs[indices_replaced] = self.tokenizer.convert_tokens_to_ids(self.tokenizer.mask_token)

        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & span_masked_indices & ~indices_replaced
        random_words = torch.randint(len(self.tokenizer), labels.shape, dtype=torch.long)
        inputs[indices_random] = random_words[indices_random]

        return inputs, labels

In [ ]:
data_collator = SpanDataCollator(tokenizer=tokenizer, mlm_probability=0.10, max_span_length=3)

In [ ]:
# data_collator = DataCollatorForWholeWordMask(
#     tokenizer=tokenizer,
#     mlm=True,
#     mlm_probability=0.15
# )

In [ ]:
from transformers import TrainerCallback

In [ ]:
class SmartPrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step % 400 == 0:
            print(f"\n🔮 --- ПРОВЕРКА НА ШАГЕ {state.global_step} ---")
            samples = [
                "[CTX_CHURCH] во имѧ ѿц҃а и [MASK] и ст҃го дх҃а",
                "[CTX_DAILY] поклоно ѿ онѳима ко [MASK]",
                "[CTX_LEGAL] а посулов бояром не [MASK]",
                "[CTX_LIT] не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть",
                "[CTX_EPIC] гой еси ты добрый [MASK]",
                "[CTX_SCIENCE] а ѿ тоя болезни дай ему пити [MASK]"
            ]
            device = kwargs['model'].device
            kwargs['model'].eval()
            with torch.no_grad():
                for text in samples:
                    inputs = tokenizer(text, return_tensors="pt").to(device)
                    mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
                    if len(mask_token_index) > 0:
                        outputs = kwargs['model'](**inputs)
                        top_3_tokens = torch.topk(outputs.logits[0, mask_token_index, :], 3, dim=1).indices[0].tolist()
                        decoded = [tokenizer.decode([t]).replace("##", "") for t in top_3_tokens]
                        cat = text.split(']')[0] + ']'
                        print(f"📝 {cat:<13} | {text.replace(cat, '').strip()}  ->  {decoded}")
            kwargs['model'].train()
            print("----------------------------------------------\n")

In [ ]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    # Оставляем только топ-5 предсказаний, чтобы не перегружать оперативную память (RAM)
    top_k_logits = torch.topk(logits, k=5, dim=-1).indices
    return top_k_logits

In [ ]:
import numpy as np

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    mask = labels != -100
    labels = labels[mask]
    preds = preds[mask]

    top1_acc = np.mean(preds[:, 0] == labels)
    top3_acc = np.mean(np.any(preds[:, :3] == labels[:, None], axis=1))
    top5_acc = np.mean(np.any(preds[:, :5] == labels[:, None], axis=1))

    return {
        "top1_accuracy": top1_acc,
        "top3_accuracy": top3_acc,
        "top5_accuracy": top5_acc,
    }

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    overwrite_output_dir=True,
    num_train_epochs=15,
    per_device_train_batch_size=64,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    eval_steps=400,
    save_steps=400,
    save_total_limit=2,
    logging_steps=100,

    # 🔥 ВАЖНО: Отключили prediction_loss_only, чтобы считались метрики
    prediction_loss_only=False,

    fp16=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=1000,
    weight_decay=0.01,
    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    callbacks=[SmartPrinterCallback()],

    # 🔥 Добавили функции для подсчета метрик
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Top1 Accuracy,Top3 Accuracy,Top5 Accuracy
400,7.466300,7.504405,0.129388,0.184902,0.210291
800,7.392100,7.340801,0.138475,0.194471,0.219862
1200,6.761500,6.633702,0.172317,0.230355,0.256682
1600,6.250000,6.154231,0.188625,0.252541,0.281951
2000,5.866300,5.744744,0.205428,0.276858,0.312477
2400,5.420800,5.288718,0.228874,0.314152,0.356998
2800,5.072900,4.937914,0.258762,0.354823,0.401506
3200,4.795700,4.661877,0.284359,0.391783,0.440214
3600,4.570500,4.444129,0.313993,0.423074,0.470898
4000,4.406700,4.293180,0.331797,0.444127,0.492199



🔮 --- ПРОВЕРКА НА ШАГЕ 400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['и', '.', ',']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  ['.', 'и', ',']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['.', 'а', ',']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['.', 'и', ',']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['.', ',', 'и']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити [MASK]  ->  ['.', 'а', ',']
----------------------------------------------


🔮 --- ПРОВЕРКА НА ШАГЕ 400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['и', '.', ',']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  ['.', 'и', ',']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['.', 'а', ',']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['.', 'и', ',']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['.', ',', 'и']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити [MASK]  ->  ['.', 

Step,Training Loss,Validation Loss,Top1 Accuracy,Top3 Accuracy,Top5 Accuracy
400,7.466300,7.504405,0.129388,0.184902,0.210291
800,7.392100,7.340801,0.138475,0.194471,0.219862
1200,6.761500,6.633702,0.172317,0.230355,0.256682
1600,6.250000,6.154231,0.188625,0.252541,0.281951
2000,5.866300,5.744744,0.205428,0.276858,0.312477
2400,5.420800,5.288718,0.228874,0.314152,0.356998
2800,5.072900,4.937914,0.258762,0.354823,0.401506
3200,4.795700,4.661877,0.284359,0.391783,0.440214
3600,4.570500,4.444129,0.313993,0.423074,0.470898
4000,4.406700,4.293180,0.331797,0.444127,0.492199



🔮 --- ПРОВЕРКА НА ШАГЕ 4400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['сн҃а', 'оц҃а', 'ст҃го']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  [':', 'смену', 'фоносу']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['.', 'имати', 'судити']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['и', 'на', 'в']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['молодец', '!', 'конь']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити [MASK]  ->  ['.', ';', ':']
----------------------------------------------


🔮 --- ПРОВЕРКА НА ШАГЕ 4400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['сн҃а', 'оц҃а', 'ст҃го']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  [':', 'смену', 'фоносу']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['.', 'имати', 'судити']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['и', 'на', 'в']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['молодец', 

There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


TrainOutput(global_step=5415, training_loss=5.486750203816844, metrics={'train_runtime': 4412.0777, 'train_samples_per_second': 156.997, 'train_steps_per_second': 1.227, 'total_flos': 2.04376981890816e+16, 'train_loss': 5.486750203816844, 'epoch': 15.0})

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

('/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/tokenizer_config.json',
 '/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/special_tokens_map.json',
 '/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/vocab.txt',
 '/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/added_tokens.json',
 '/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/tokenizer.json')

In [ ]:
import math

# Оценка на валидационной выборке
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])

print(f"📊 Результаты после 15 эпох:")
print(f"Финишый Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {perplexity:.2f}")

if perplexity < 20:
    print("🏆 Модель великолепно выучила структуру языка!")
elif perplexity < 50:
    print("📈 Хороший результат, модель понимает контекст.")
else:
    print("⚠️ Модели было сложно. Возможно, нужно больше данных или слоев.")

📊 Результаты после 15 эпох:
Финишый Loss: 4.1090
Perplexity: 60.89
⚠️ Модели было сложно. Возможно, нужно больше данных или слоев.


In [ ]:
from transformers import pipeline

print("🔍 Загрузка обученной модели для финального теста...")
fill_mask = pipeline(
    "fill-mask",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=0,
)

# Тесты для каждой категории
final_tests = [
    {
        "category": "⛪️ [CTX_CHURCH] (Ожидаем: сына / отца / бога / духа)",
        "text": "[CTX_CHURCH] Во имя отца и [MASK] и святаго духа."
    },
    {
        "category": "🏡 [CTX_DAILY] (Ожидаем: господину / брату / юрью)",
        "text": "[CTX_DAILY] Поклонъ ѿ бориса ко [MASK] съ бг҃омъ."
    },
    {
        "category": "⚖️ [CTX_LEGAL] (Ожидаем: винити / судити / имати / дати)",
        "text": "[CTX_LEGAL] Аже оубиеть моужь мужа, то мьстити брату, а посулов не [MASK] ."
    },
    {
        "category": "📚 [CTX_LIT] (Ожидаем: словесы / дѣлы)",
        "text": "[CTX_LIT] Не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть."
    },
    {
        "category": "⚔️ [CTX_EPIC] (Ожидаем: молодец / богатырь / конь)",
        "text": "[CTX_EPIC] Гой еси ты добрый [MASK] , куда путь держишь?"
    },
    {
        "category": "🌿 [CTX_SCIENCE] (Ожидаем: зеліе / траву / воду)",
        "text": "[CTX_SCIENCE] А ѿ тоя болезни дай ему пити [MASK] , и тако исцелеет."
    }
]

print("\n" + "=" * 60)
print("🏆 ФИНАЛЬНЫЙ ЭКЗАМЕН MINI-BERT (ВСЕ КАТЕГОРИИ)")
print("=" * 60)

for test in final_tests:
    print(f"\n🔹 {test['category']}")
    print(f"Текст: {test['text']}")
    results = fill_mask(test["text"])
    for i, res in enumerate(results[:3]):
        print(f"  {i+1}. {res['token_str']:<12} (Уверенность: {res['score']*100:.1f}%)")

🔍 Загрузка обученной модели для финального теста...

🏆 ФИНАЛЬНЫЙ ЭКЗАМЕН MINI-BERT (ВСЕ КАТЕГОРИИ)

🔹 ⛪️ [CTX_CHURCH] (Ожидаем: сына / отца / бога / духа)
Текст: [CTX_CHURCH] Во имя отца и [MASK] и святаго духа.
  1. сына         (Уверенность: 76.3%)
  2. отца         (Уверенность: 6.6%)
  3. духа         (Уверенность: 4.0%)

🔹 🏡 [CTX_DAILY] (Ожидаем: господину / брату / юрью)
Текст: [CTX_DAILY] Поклонъ ѿ бориса ко [MASK] съ бг҃омъ.
  1. василью      (Уверенность: 11.4%)
  2. мнѣ          (Уверенность: 7.1%)
  3. гн҃у         (Уверенность: 4.9%)

🔹 ⚖️ [CTX_LEGAL] (Ожидаем: винити / судити / имати / дати)
Текст: [CTX_LEGAL] Аже оубиеть моужь мужа, то мьстити брату, а посулов не [MASK] .
  1. имати        (Уверенность: 51.9%)
  2. надобѣ       (Уверенность: 7.7%)
  3. просити      (Уверенность: 5.4%)

🔹 📚 [CTX_LIT] (Ожидаем: словесы / дѣлы)
Текст: [CTX_LIT] Не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть.
  1. ,            (Уверенность: 13.4%)
  2. и            (Уверенн

In [63]:
HF_USERNAME = "AlexSychovUN"

In [64]:
from huggingface_hub import notebook_login

notebook_login()

In [62]:
print("\n🚀 ПУШИМ MINI-BERT (SPAN MASKING)...")
bert_model = BertForMaskedLM.from_pretrained(MODEL_DIR)
bert_tokenizer = BertTokenizerFast.from_pretrained(MODEL_DIR)

bert_repo = f"{HF_USERNAME}/mini-bert-ancient-rus-span-collator-v1"
bert_model.push_to_hub(bert_repo)
bert_tokenizer.push_to_hub(bert_repo)
print(f"✅ BERT успешно загружен: https://huggingface.co/{bert_repo}")


🚀 ПУШИМ MINI-BERT (SPAN MASKING)...


HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-699eb38e-5a7a4dfe4fbb53d97cde907c;010890a0-c593-4a6e-aa81-92e0458962aa)

Invalid username or password.